# Install Dependencies

**CRITICAL: El orden de instalación importa en Blackwell (sm_120).**

1. PyTorch con cu128 **PRIMERO** (stable no soporta sm_120)
2. triton >= 3.3.1
3. bitsandbytes
4. unsloth (trae peft, transformers, accelerate)
5. trl, datasets, anthropic, pydantic

Si se instala en otro orden, vLLM o unsloth pueden reinstalar torch con cu126 y romper Blackwell.

**Nota Windows:** usar `dataset_num_proc=1` en todos los scripts de training.

## 1. Install PyTorch (cu128 — Blackwell)

In [ ]:
# If stable PyTorch already supports cu128 by the time you run this,
# you can remove --pre and use the stable index instead.
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

## 2. Verify PyTorch + Blackwell

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available after install!"
cap = torch.cuda.get_device_capability(0)
print(f"PyTorch {torch.__version__}")
print(f"CUDA {torch.version.cuda}")
print(f"Compute capability: {cap[0]}.{cap[1]} (sm_{cap[0]}{cap[1]}0)")

if cap[0] < 12:
    print("\n⚠ sm_120 NOT detected. This torch build may not support Blackwell.")
    print("  Ensure you installed from the cu128 nightly index.")
else:
    print("\n✓ Blackwell OK")

## 3. Install triton

In [ ]:
# triton >= 3.3.1 for Blackwell kernel compilation.
# If this fails on Windows, it's OK — we'll use attn_implementation="sdpa" instead.
!pip install "triton>=3.3.1"

## 4. Install bitsandbytes

In [ ]:
!pip install bitsandbytes

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes: {bnb.__version__}")
except Exception as e:
    print(f"⚠ bitsandbytes import failed: {e}")
    print("  4-bit QLoRA may not work. Fall back to load_in_8bit or try building from source.")

## 5. Install unsloth

In [ ]:
!pip install unsloth

try:
    from unsloth import FastLanguageModel
    print("✓ unsloth imported successfully")
except Exception as e:
    print(f"⚠ unsloth import failed: {e}")
    print("  Check https://docs.unsloth.ai/basics/installation for troubleshooting.")

## 6. Install remaining dependencies

In [ ]:
!pip install trl datasets anthropic pydantic

## 7. Full Dependency Verification

In [ ]:
import importlib

deps = {
    "torch": "torch",
    "unsloth": "unsloth",
    "trl": "trl",
    "transformers": "transformers",
    "datasets": "datasets",
    "bitsandbytes": "bitsandbytes",
    "peft": "peft",
    "accelerate": "accelerate",
    "anthropic": "anthropic",
    "pydantic": "pydantic",
    "triton": "triton",
}

all_ok = True
print(f"{'Package':<20} {'Version':<20} {'Status'}")
print("-" * 50)
for name, module in deps.items():
    try:
        m = importlib.import_module(module)
        v = getattr(m, "__version__", "?")
        print(f"{name:<20} {v:<20} ✓")
    except ImportError:
        print(f"{name:<20} {'—':<20} ✗ MISSING")
        all_ok = False

print()
if all_ok:
    print("✓ All dependencies installed. Ready for training.")
else:
    print("✗ Some dependencies missing. Review errors above.")

## 8. Quick unsloth + Qwen3-8B Load Test

Verifica que unsloth puede cargar el modelo base. Esto descarga ~5GB la primera vez.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

free, total = torch.cuda.mem_get_info(0)
used = (total - free) / 1e9
print(f"\nModel loaded. VRAM used: {used:.1f} GB / {total/1e9:.1f} GB")
print(f"VRAM free: {free/1e9:.1f} GB")
print("\n✓ Qwen3-8B loads correctly with unsloth. Ready for fine-tuning.")

# Cleanup
del model, tokenizer
torch.cuda.empty_cache()